In [1]:
import torch
import numpy as np
import torch.nn as nn
import torch.nn.functional as F

from fastdtw import fastdtw
from torch.utils.data import Dataset, DataLoader

In [2]:
Q = torch.load("data/queries.pt")
C = torch.load("data/corpuses.pt")
dist = torch.load("data/sims.pt")
Q.shape, C.shape, dist.shape

(torch.Size([1000, 6, 4000]),
 torch.Size([1000, 20, 4000]),
 torch.Size([1000, 1000]))

# Dataset

In [3]:
import sys
sys.path.append("../neuts/")

import tools.sampling_methods as sm
from geo_rnns.wrloss import WeightedRankingLoss

class DTWDataset(Dataset):
    def __init__(self, Q, C, dist, per_q_expl=200):
        self.Q = Q
        self.C = C
        self.dist = dist
        self.per_q_expl = per_q_expl
        self.batches = []
        self.create_batches()

    def create_batches(self):
        for i in range(len(self.Q)):
            distances = self.dist[i]
            sortidx = torch.argsort(distances)
            top_distances = distances[sortidx[:self.per_q_expl]]
            bot_distances = distances[sortidx[-self.per_q_expl:]]

            batch = [i, sortidx[:self.per_q_expl], sortidx[-self.per_q_expl:], top_distances, bot_distances]
            self.batches.append(batch)

    def __len__(self):
        return len(self.Q)

    def __getitem__(self, idx):
        query_idx, pos_corpus_idxs, neg_corpus_idxs, pos_distances, neg_distances = self.batches[idx]
        query = self.Q[query_idx]
        pos_corpus = self.C[pos_corpus_idxs]
        neg_corpus = self.C[neg_corpus_idxs]
        return query, pos_corpus, neg_corpus, pos_distances, neg_distances

In [70]:
device = "cuda:0"
model = nn.GRU(input_size=1000, hidden_size=128, num_layers=2, batch_first=True).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

dataset = DTWDataset(Q[:,:,::4], C[:,:,::4], dist)
loader = DataLoader(dataset, batch_size=1, shuffle=True)

In [94]:
from tqdm import tqdm

def embed(x):
    return model(x)[0][:,-1,:]

pbar = tqdm(range(10))
for epoch in pbar:
    losses = []
    for q, pc, nc, pd, nd in loader:
        q, pc, nc, pd, nd = q.to(device), pc.to(device), nc.to(device), pd.to(device), nd.to(device)

        qemb = embed(q).repeat_interleave(200, dim=0)
        pcemb = embed(pc[0])
        ncemb = embed(nc[0])

        pos_preds = torch.exp(-F.pairwise_distance(qemb, pcemb))
        neg_preds = torch.exp(-F.pairwise_distance(qemb, ncemb))

        loss = F.mse_loss(pos_preds, pd[0]) + F.mse_loss(neg_preds, nd[0])

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        losses.append(loss.item())
        pbar.set_description(f"Loss: {np.mean(losses)}")

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/10 [00:20<?, ?it/s]


KeyboardInterrupt: 

# inference

In [1]:
from train import *

In [7]:
device = "cuda:1"
model = torch.load("models/tsize3500_step10_GRU.pt", map_location=device)
model.eval()

RNNEncoder(
  (cell): GRUCell(399, 128)
)

In [8]:
import sys
sys.path.append("../../")

from opas.data import PairDatasetTest
from opas.utils import normalize

step = 10   # from file name
test_dataset = PairDatasetTest("../../final_data/audio/dataset_test.hdf5")

In [9]:
with torch.no_grad():
    C = torch.from_numpy(test_dataset.c).float()
    Cembed = []
    inner_batch_size = 200

    for batch in tqdm(range(0, len(C), inner_batch_size), disable=True):
        c = C[batch:batch+inner_batch_size].to(device)
        c = model(c[:,:,::step])
        Cembed.append(c)
    C = torch.vstack(Cembed)

    netscores = []
    true_labels = []
    for i in tqdm(range(len(test_dataset))):    
        q, l = test_dataset[i]
        q, l = q.to(device), l.to(device)
        q = q[:, ::step].unsqueeze(0)
        q = model(q)
        scores = F.pairwise_distance(q, C)
        netscores.append(scores.to('cpu'))
        true_labels.append(l.to('cpu'))
    
    netscores = torch.vstack(netscores)
    true_labels = torch.vstack(true_labels).squeeze() # 

    ranking = netscores.argsort(dim=1, descending=True)
    ranked_output = torch.gather(true_labels, dim=1, index=ranking)

    mRR = (1 / (ranked_output.argmax(dim=1) + 1)).mean().item()

    mAP = (torch.cumsum(ranked_output, dim=1) * ranked_output).float()
    mAP /= (torch.arange(ranked_output.shape[1]) + 1)
    mAP /= torch.sum(ranked_output, dim=1, keepdim=True)
    mAP = mAP.sum(dim=1).mean().item()

100%|██████████| 3040/3040 [00:14<00:00, 208.71it/s]


In [10]:
netscores.shape, true_labels.shape

(torch.Size([3040, 4864]), torch.Size([3040, 4864]))

In [11]:
print(f"{mAP:.4f}\t{mRR:.4f}")

0.0053	0.0024
